In [1]:
import polars as pl
import plotly.express as px
from constants import DATA_FOLDER, DATA_FILE, holidays, long_holidays
from datetime import datetime, date, time
from helpers import clean_df, create_travel_time_column, create_time_features
import lightgbm as gbm
from sklearn.model_selection import KFold, train_test_split


pl.Config.set_tbl_rows(30)

polars.config.Config

In [2]:
df_raw = pl.scan_parquet(DATA_FILE)
df = df_raw.filter(pl.col("RouteID") == 1728).pipe(clean_df).collect()

In [3]:
df_target = create_travel_time_column(df, "新竹轉運站", "龍潭運動公園", "返程").pipe(
    create_time_features
)

In [4]:
df_target.head()

Route,Plate,Direction,Stop,StopSeq,Event,Time,DepartureTime,Route_right,Direction_right,Stop_right,StopSeq_right,Event_right,ArrivalTime,Duration_min,is_rush_hour,day_of_week
str,str,cat,str,i32,cat,datetime[μs],datetime[μs],str,cat,str,i32,cat,datetime[μs],f64,cat,cat
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:01:29,2025-02-28 06:01:29,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""",2025-02-28 06:45:15,43.766667,"""not_rush""","""Fri"""
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:59:02,2025-02-28 06:59:02,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""",2025-02-28 07:41:21,42.316667,"""not_rush""","""Fri"""
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 08:01:19,2025-02-28 08:01:19,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""",2025-02-28 08:52:13,50.9,"""morning_rush""","""Fri"""
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 09:00:50,2025-02-28 09:00:50,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""",2025-02-28 10:06:22,65.533333,"""morning_rush""","""Fri"""
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 10:28:42,2025-02-28 10:28:42,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""",2025-02-28 11:40:29,71.783333,"""not_rush""","""Fri"""


In [5]:
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

for day in day_order:
    px.box(
        df_target.filter(pl.col("day_of_week") == day),
        x="is_rush_hour",
        y="Duration_min",
        title=f"{day}",
    ).show()

In [6]:
px.histogram(
    df_target.filter(
        pl.col("day_of_week").is_in(["Sat", "Sun"]),
        # pl.col("is_rush_hour") == "evening_rush",
    ),
    x="Duration_min",
    nbins=30,
)


In [7]:
df_target.filter(
    pl.col("day_of_week") == "Sat",
    # pl.col("is_rush_hour") == "not_rush",
).select("Duration_min").describe([0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])

statistic,Duration_min
str,f64
"""count""",736.0
"""null_count""",0.0
"""mean""",58.066372
"""std""",7.445774
"""min""",17.0
"""5%""",45.783333
"""10%""",48.166667
"""25%""",53.85
"""50%""",57.95


In [8]:
df.filter(
    pl.col("Plate") == "KKB-1692",
    pl.col("Time").is_between(date(2025, 12, 6), datetime(2025, 12, 6, 23, 59, 59)),
    pl.col("Event") == "離站",
).sort("Time")

Route,Plate,Direction,Stop,StopSeq,Event,Time
str,str,cat,str,i32,cat,datetime[μs]
"""17280""","""KKB-1692""","""返程""","""新竹轉運站""",1,"""離站""",2025-12-06 06:00:12
"""17280""","""KKB-1692""","""返程""","""公園站""",2,"""離站""",2025-12-06 06:00:50
"""17280""","""KKB-1692""","""返程""","""新竹轉運站""",1,"""離站""",2025-12-06 09:01:53
"""17280""","""KKB-1692""","""返程""","""公園站""",2,"""離站""",2025-12-06 09:04:10
"""17280""","""KKB-1692""","""返程""","""二分局""",3,"""離站""",2025-12-06 09:06:50
"""17280""","""KKB-1692""","""返程""","""馬偕醫院""",4,"""離站""",2025-12-06 09:08:35
"""17280""","""KKB-1692""","""返程""","""清華大學""",5,"""離站""",2025-12-06 09:14:27
"""17280""","""KKB-1692""","""返程""","""陽明交大光復路""",6,"""離站""",2025-12-06 09:16:08
"""17280""","""KKB-1692""","""返程""","""科技生活館""",7,"""離站""",2025-12-06 09:24:15


In [9]:
# try running lightbgm model to see if it works well on 1728 with the current features
df_train = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 2, 25), datetime(2025, 11, 30, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)
df_test = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)
# remove faulty data points
# only 1 point removed

In [10]:
X_train = df_train.drop("Duration_min").to_pandas()
y_train = df_train.select("Duration_min").to_pandas()
X_test = df_test.drop("Duration_min").to_pandas()
y_test = df_test.select("Duration_min").to_pandas()

In [11]:
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [12]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 182
[LightGBM] [Info] Number of data points in the train set: 5386, number of used features: 2
[LightGBM] [Info] Start training from score 58.917335


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [13]:
y_pred = model.predict(X_test)

In [14]:
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [15]:
output.select(
    (abs(pl.col("prediction") - pl.col("label")) < 10).mean().alias("within_10_min"),
    (abs(pl.col("prediction") - pl.col("label")) < 5).mean().alias("within_5_min"),
)

within_10_min,within_5_min
f64,f64
0.87234,0.628478


# Baseline prediction accuracy
- With only basic features (day of the week (Mon, Tue ...), time in a day (in minutes)), we managed to gain **87.23%** accuracy.
- A prediction is classified as accurate if it is within 10 minutes from the true travel time.

## Let's try with full features
1. day of week
2. time
3. holidays
4. long holidays

In [16]:
df_train = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 2, 25), datetime(2025, 11, 30, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        ),
        pl.col("Time").dt.date().is_in(holidays).alias("is_holiday"),
        pl.col("Time").dt.date().is_in(long_holidays).alias("is_long_holiday"),
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "is_holiday",
            "is_long_holiday",
            "Duration_min",
        ]
    )
)

df_test = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        ),
        pl.col("Time").dt.date().is_in(holidays).alias("is_holiday"),
        pl.col("Time").dt.date().is_in(long_holidays).alias("is_long_holiday"),
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "is_holiday",
            "is_long_holiday",
            "Duration_min",
        ]
    )
)

In [17]:
X_train = df_train.drop("Duration_min").to_pandas()
y_train = df_train.select("Duration_min").to_pandas()
X_test = df_test.drop("Duration_min").to_pandas()
y_test = df_test.select("Duration_min").to_pandas()

In [18]:
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [19]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000620 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 186
[LightGBM] [Info] Number of data points in the train set: 5386, number of used features: 4
[LightGBM] [Info] Start training from score 58.917335


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
y_pred = model.predict(X_test)

In [21]:
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [22]:
output.select(
    (abs(pl.col("prediction") - pl.col("label")) < 10).mean().alias("within_10_min"),
    (abs(pl.col("prediction") - pl.col("label")) < 5).mean().alias("within_5_min"),
)

within_10_min,within_5_min
f64,f64
0.869067,0.628478


In [23]:
feat_importance = pl.DataFrame(
    {
        "feature": X_train.columns,
        "importance": model.booster_.feature_importance(importance_type="gain"),
    }
).sort("importance", descending=True)

feat_importance

feature,importance
str,f64
"""minutes_from_midnight""",3.3995e6
"""day_of_week""",346109.161107
"""is_holiday""",47599.306293
"""is_long_holiday""",2474.939041


In [24]:
display(df_train.select(pl.col("is_holiday").value_counts()))
display(df_train.select(pl.col("is_long_holiday").value_counts()))

is_holiday
struct[2]
"{false,5026}"
"{true,360}"


is_long_holiday
struct[2]
"{false,5041}"
"{true,345}"


## result
- is_holiday and is_long_holiday seem to add noise to the model and actually worsen the model performance
- try evaluating the models during regular days (not long holidays) as intuitively travel time during long holidays is a lot harder to predict

In [25]:
df_train = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 2, 25), datetime(2025, 11, 30, 23, 59, 59)
        ),
        ~pl.col("Time").dt.date().is_in(long_holidays),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)
df_test = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)
        ),
        ~pl.col("Time").dt.date().is_in(long_holidays),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)

In [26]:
X_train = df_train.drop("Duration_min").to_pandas()
y_train = df_train.select("Duration_min").to_pandas()
X_test = df_test.drop("Duration_min").to_pandas()
y_test = df_test.select("Duration_min").to_pandas()

In [27]:
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [28]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 177
[LightGBM] [Info] Number of data points in the train set: 5041, number of used features: 2
[LightGBM] [Info] Start training from score 58.881928


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [29]:
y_pred = model.predict(X_test)

In [30]:
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [31]:
output.select(
    (abs(pl.col("prediction") - pl.col("label")) < 10).mean().alias("within_10_min"),
    (abs(pl.col("prediction") - pl.col("label")) < 5).mean().alias("within_5_min"),
)

within_10_min,within_5_min
f64,f64
0.870704,0.617021


## finding
- basically the same accuracy as with the holiday rows included
- holidays are not useful features in this problem and only add noise

# Baseline guessing

In [38]:
test = df_target.filter(
    pl.col("Duration_min").is_between(20, 120),
    pl.col("Time").is_between(date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)),
).select(
    "Duration_min",
)
n = test.height

In [ ]:
output = pl.DataFrame(
    {"prediction": [58] * n, "label": test}
)  # 58 is the mean and median of travel time
output.select(
    (abs(pl.col("prediction") - pl.col("label")) < 10).mean().alias("within_10_min"),
    (abs(pl.col("prediction") - pl.col("label")) < 5).mean().alias("within_5_min"),
)

within_10_min,within_5_min
f64,f64
0.690671,0.378069
